In [2]:
# ===================== DAY 1: data, tokenizer, batches =====================
import torch

torch.manual_seed(1337)

# --- hyperparameters (we'll grow this list as we go) ---
batch_size = 4   # how many independent sequences we process in parallel
block_size = 8   # max context length the model ever sees

# --- 1. load the data ---
# first run in a notebook:  !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters:", len(text))  # expect 1115394

# --- 2. vocabulary: every unique character is a token ---
chars = sorted(list(set(text)))
vocab_size = len(chars)  # expect 65
print(''.join(chars), vocab_size)

# --- 3. tokenizer: char <-> int ---
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]             # string -> list of ints
decode = lambda l: ''.join([itos[i] for i in l])    # list of ints -> string
assert decode(encode("hii there")) == "hii there"   # round trip must be lossless

# --- 4. whole dataset as one long tensor of ints ---
data = torch.tensor(encode(text), dtype=torch.long)  # shape (1115394,)

# --- 5. train/val split: first 90% train, last 10% val ---
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# --- 6. batching ---
# A chunk of block_size+1 chars gives block_size training examples:
# x = chars [0..7], y = chars [1..8]; at position t, input is x[:t+1], target is y[t].
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))   # random start positions
    x = torch.stack([d[i:i + block_size] for i in ix])        # (B, T) inputs
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])  # (B, T) targets, shifted by 1
    return x, y

xb, yb = get_batch('train') # xb training set and yb target. 
print(xb.shape, yb.shape)  # both torch.Size([4, 8])
# ===========================================================================

length of dataset in characters: 1115393

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz 65
torch.Size([4, 8]) torch.Size([4, 8])


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

         
        self.embedding  = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # TODO: logits from the table, shape (B, T, C)
        logits = self.embedding(idx)
        


        # TODO: if targets is None -> loss = None
        #       else reshape logits and targets (your answer to question 3), compute cross_entropy
        if(targets == None):
            loss = None   
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)


        return logits, loss


    def generate(self, idx, max_new_tokens):
        # TODO: loop max_new_tokens times:
        #   get logits, take the last position only, softmax, multinomial, append to idx
        for _ in range(max_new_tokens):
            logits, loss = self.forward(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, ix), dim=1)
            

        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)                     # parameter 1                                    # parameter 2
print(      decode(     m.generate(idx = torch.zeros((1,1), dtype=torch.long),      max_new_tokens=100)[0].tolist())     )

torch.Size([32, 65])

lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


In [ ]:
# TODO: create the optimizer (AdamW, lr=1e-3) from the model's parameters
# nn modules knows its parameters by nn.embedding() where nn is torch.nn

batch_size = 32

for step in range(100):   # start small, then raise it
    # TODO: 1. get a train batch
    # TODO: 2. forward pass -> logits, loss
    # TODO: 3. clear old gradients
    # TODO: 4. backward pass
    # TODO: 5. optimizer step
    pass

print(loss.item())
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))
           

tensor([[0]])